# 08 Compute family features

目标：在全部筛选合格WFICN月宇宙中计算五个家族特征，除fund_no外排除目标基金。

- 输入：05、06、07基金输出
- 输出：family_features_raw.parquet及家族规模审计
- 主键：wficn+month
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：05–07
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import numpy as np, pandas as pd, polars as pl
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path, upstream_path
from src.schema_utils import assert_unique

basic=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"05_aggregate_shareclasses_and_fund_features","data/processed/fund/fund_month_basic_features.parquet"))
mom=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/processed/fund/fund_alpha_momentum_raw.parquet")).select(["wficn","month","F_r12_2"])
mgmt=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"07_build_management_company_mapping","data/processed/fund/wficn_month_management_company.parquet")).select(["wficn","month","mgmt_cd"])
panel=basic.join(mom,on=["wficn","month"],how="left").join(mgmt,on=["wficn","month"],how="left").to_pandas()
rows=[]
for (month,code),g in panel.groupby(["month","mgmt_cd"],dropna=False):
    for _,target in g.iterrows():
        others=g[g["wficn"]!=target["wficn"]]
        valid_mom=others["tna"].notna()&others["F_r12_2"].notna()&(others["tna"]>0)
        valid_flow=others["tna"].notna()&others["flow"].notna()&(others["tna"]>0)
        rows.append({
            "wficn":target["wficn"],"month":month,
            "Family_TNA":others["tna"].sum(min_count=1) if pd.notna(code) else np.nan,
            "fund_no":len(g) if pd.notna(code) else np.nan,
            "Family_r12_2":np.average(others.loc[valid_mom,"F_r12_2"],weights=others.loc[valid_mom,"tna"]) if valid_mom.any() else np.nan,
            "Family_age":others["age"].max() if len(others) else np.nan,
            "Family_flow":np.average(others.loc[valid_flow,"flow"],weights=others.loc[valid_flow,"tna"]) if valid_flow.any() else np.nan,
        })
result=pl.from_pandas(pd.DataFrame(rows) if rows else pd.DataFrame(columns=["wficn","month","Family_TNA","fund_no","Family_r12_2","Family_age","Family_flow"])).with_columns(
    pl.col("month").cast(pl.Date,strict=False),
    *[pl.col(c).cast(pl.Float64,strict=False) for c in ["Family_TNA","fund_no","Family_r12_2","Family_age","Family_flow"]],
); assert_unique(result,["wficn","month"],"family features")
out=output_path(ROOT,RUN_MODE,"08_compute_family_features","data/processed/fund/family_features_raw.parquet")
audit=output_path(ROOT,RUN_MODE,"08_compute_family_features","data/audit/08_family_size_distribution.csv")
write_parquet_atomic(result,out); write_csv_atomic(result.group_by("fund_no").len(),audit)
INPUTS_USED=["fund_month_basic_features.parquet","fund_alpha_momentum_raw.parquet","wficn_month_management_company.parquet"]
OUTPUTS_WRITTEN=[str(out.relative_to(ROOT)),str(audit.relative_to(ROOT))]; INPUT_COUNTS={"eligible_fund_months":panel.shape[0]}; OUTPUT_COUNTS={"family_rows":result.height}
PRIMARY_KEY=["wficn","month"]; DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=0; EXCEPTION_FILE=str(audit.relative_to(ROOT)); STAGE_STATUS="passed"
STAGE_NOTES=["Family universe was not reduced by holdings, stock-feature, or label availability."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "08_compute_family_features", stage="08", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
